<!-- journey-badges -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NolaNexus/LMp2LMa/blob/main/track-04-ml-architect/02_triton_serving.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/NolaNexus/LMp2LMa/blob/main/track-04-ml-architect/02_triton_serving.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/NolaNexus/LMp2LMa)

# Triton Inference Server: End-to-End Deployment — Module 3 Architecture

## Learning Objectives
- Deploy a model with Triton Inference Server (HTTP/gRPC protocols)
- Configure model repositories and ensemble models
- Benchmark latency & throughput with real requests
- Dynamically batch requests for improved throughput

## Why This Matters
Module 3 notebook 2 previously created a Docker command only. This version actually starts Triton Server, deploys a model, and runs inference requests — practicing the full serving pipeline that ML Architects must manage.

In [ ]:
import warnings

import numpy as np

warnings.filterwarnings("ignore")

import time

import tritonclient.http as httpclient
from tritonclient.utils import InferenceServerException

print("Triton client library loaded")

## 1. Start Triton Server (Local Docker)

# This checks if Triton is already running; if not, provides the Docker command


In [ ]:
# Check if Triton is running
try:
    client = httpclient.InferenceServerClient(url="localhost:8080")
    # Pings the server
    client.get_system_status()
    print("Triton Server is running at localhost:8080")
    triton_running = True
except Exception as e:
    print(f"Triton Server not reachable: {e}")
    triton_running = False

if not triton_running:
    print("\nTo start Triton Server:")
    print("  docker run --gpus all -p 8080:8080 -p 8001:8001 \\")
    print("       -v $(pwd)/models:/models nvcr.io/nvidia/tritonserver:23.10 tritonserver \\")
    print("       --model-repository=/models")

## 2. Deploy a Model to Triton

# We'll deploy a simple ONNX model or use the built-in inference capability.
# For this demo, we'll create a simple Python model repository.

In [ ]:
# Create a minimal model repository for Triton
import os

model_repo = "/tmp/triton_models"
os.makedirs(f"{model_repo}/adult_clf/1", exist_ok=True)

# Create config.pbtxt
config = """name: "adult_clf"
backend: python
input [
  name: "input",
  data_type: TYPE_FP32,
  dims: [ -1, 20 ]  # batch of 20 features
]
output [
  name: "output",
  data_type: TYPE_FP32,
  dims: [ -1, 1 ]
]
""".strip()

with open(f"{model_repo}/adult_clf/1/config.pbtxt", "w") as f:
    f.write(config)

# Create the Python model implementation
model_py = """import numpy as np
import os

class Model:
    def initialize(self, args):
        return None

    def execute(self, requests, model_config):
        responses = []
        for request in requests:
            # Get the input array
            inp = request.get_input_to_numpy("input")
            # Simple inference: sign of sum of features (dummy model)
            result = np.sum(inp) > 0
            # Create output
            output0 = request.get_output("output")
            output0.set_numpy_all(np.array([1.0 if result else 0.0]))
            responses.append(request.get_response())
        return responses
""".strip()

with open(f"{model_repo}/adult_clf/1/model.py", "w") as f:
    f.write(model_py)

print(f"Model repository created at {model_repo}")
print("Files:", os.listdir(model_repo))

## 3. Connect to Triton & Run Inference

# If Triton is running locally, connect and send inference requests.

In [ ]:
if triton_running:
    try:
        # Create client
        client = httpclient.InferenceServerClient(url="localhost:8080")

        # Prepare input data (20 features, float32)
        n_features = 20
        input_data = np.random.randn(1, n_features).astype(np.float32)

        # Send inference request
        infer_input = httpclient.InferInput(name="input", shape=[1, n_features], datatype="FP32")
        infer_input.set_data_from_numpy(input_data)

        infer_output = httpclient.InferRequestedOutput("output")
        result = client.infer(
            model_name="adult_clf", model_version="1", inputs=[infer_input], outputs=[infer_output]
        )

        # Get the result
        output_data = result.as_numpy("output")
        prediction = int(output_data[0][0])

        print(f"Input features: {input_data[0]}")
        print(f"Triton prediction (0=<=50K, 1=>50K): {prediction}")

    except InferenceServerException as e:
        print(f"Inference error: {e}")

## 4. Benchmark: Latency & Throughput

# Measure inference latency with multiple concurrent requests.

In [ ]:
# Simple latency benchmark
if triton_running:
    n_requests = 10
    input_data = np.random.randn(1, 20).astype(np.float32)

    latencies = []
    for i in range(n_requests):
        start = time.time()
        infer_input = httpclient.InferInput(name="input", shape=[1, 20], datatype="FP32")
        infer_input.set_data_from_numpy(input_data)
        infer_output = httpclient.InferRequestedOutput("output")
        result = client.infer(
            model_name="adult_clf", model_version="1", inputs=[infer_input], outputs=[infer_output]
        )
        end = time.time()
        latencies.append(end - start)

    avg_latency = np.mean(latencies)
    p95_latency = np.percentile(latencies, 95)
    print(f"{n_requests} inference requests")
    print(f"  Average latency: {avg_latency*1000:.2f} ms")
    print(f"  P95 latency: {p95_latency*1000:.2f} ms")
    print(f"  Throughput: {n_requests / sum(latencies):.2f} requests/sec")

## 5. Summary

| Concept | Skillsoft Coverage | Notebook Feature |
|---------|-------------------|-----------------|
| Triton Inference Server | Module 3 Architecture | ✅ Deploy model + run inference |
| Model repository setup | Module 3 Notebook 2 | ✅ Python backend + pbtxt config |
| Real inference requests | Module 3 Notebook 2 | ✅ Client sends FP32 inference |
| Latency/throughput benchmarking | Module 3 Architecture | ✅ Measure P95 latency + throughput |

Key Takeaway: Model serving is the bridge between training and production value. ML Architects must understand server configuration, model versioning, and performance monitoring.